# 02_pipeline — core delivery pipeline

Builds, transforms, validates, and publishes the data product defined by `01_agreement`. Run this after agreement metadata is ready and before `03_review`.

Required delivery flow: `01_agreement` → `02_pipeline` → `03_review`. Optional discovery, profiling, troubleshooting, investigation, and ad hoc analysis belongs in `99_explore`.

This template keeps source and target guardrails separate while using beginner-friendly presets:

Read source
    ↓
Validate source schema
    ↓
Monitor source data changes
    ↓
Stop if either blocking check fails
    ↓
Transform
    ↓
Validate proposed target schema
    ↓
Monitor proposed target changes
    ↓
Enforce approved active DQ rules
    ↓
Stop if any blocking guardrail fails
    ↓
Publish full target dataset
    ↓
Store profile evidence and lineage

This production pipeline reads approved active DQ rules from `METADATA_DQ_RULES` and applies them as simple aggregate guardrails before the target write. It does not filter invalid rows, create exception datasets, or partially write targets.


## 1. Runtime setup

Load the shared FabricOps environment, path configuration, and metadata routing. Keep this active so the template remains plug-and-play in Microsoft Fabric.


In [ ]:
%run 00_env_config


## 2. Imports

Import only the helpers needed for the core pipeline flow. DQ enforcement is included because approved active DQ rules are now treated as the same simple target-write guardrail pattern as schema and data-change checks.


In [ ]:
import json
from datetime import datetime, timezone

from pyspark.sql import functions as F

from fabricops_kit import (
    widget_select_agreement,
    get_selected_agreement,
    read_lakehouse_csv,
    read_lakehouse_parquet,
    read_lakehouse_excel,
    read_lakehouse_table,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
    validate_schema,
    monitor_data_changes,
    stop_if_failed,
    enforce_dq_rules,
    build_lineage_records,
)


## 3. Pipeline parameters

Set these values for the source, target, catalogue evidence, lineage evidence, and pipeline identity. The default values use the starter-kit sample table and remain run-all safe once `00_env_config` has seeded sample data.


## Choose pipeline checks

### Schema checking

#### `strict`

Use when the dataframe must exactly match the expected schema.

Default behaviour:

- Missing columns stop the pipeline.
- Datatype changes stop the pipeline.
- Unexpected new columns stop the pipeline.

#### `allow_new_columns`

Use when upstream systems may add columns.

Default behaviour:

- Missing required columns stop the pipeline.
- Datatype changes stop the pipeline.
- Unexpected new columns are allowed and reported.

#### `monitor_only`

Use when schema changes should be visible but should not stop execution.

Default behaviour:

- Report missing columns, datatype changes, and unexpected columns.
- Always return `can_continue=True`.

### Data-change monitoring

#### `changing_data`

Use for operational or transactional data that changes regularly.

Default behaviour:

- Compare with the latest successful profile.
- Row count may change by up to 50%.
- Null percentage may change by up to 20 percentage points.
- Distinct percentage may change by up to 30 percentage points.
- Numeric PSI warns at 0.10 and blocks at 0.25.
- Categorical distance warns at 0.10 and blocks at 0.25.
- Blocking drift stops publication.

#### `fixed_data`

Use for reference, historical, or controlled data that should remain stable.

Default behaviour:

- Compare with an approved baseline.
- Any row-count, null-rate, or distinct-rate change is treated strictly.
- Numeric PSI warns at 0.01 and blocks at 0.10.
- Categorical distance warns at 0.01 and blocks at 0.10.
- Blocking drift stops publication.
- The current profile does not automatically replace the approved baseline.

#### `monitor_changing_data`

Use when operational or transactional changes should be reported without blocking.

Default behaviour:

- Compare with the latest successful profile.
- Use the same thresholds as `changing_data`.
- Always return `can_continue=True`.

#### `monitor_fixed_data`

Use when reference, historical, or controlled data should be compared with an approved baseline without blocking.

Default behaviour:

- Compare with an approved baseline.
- Use the same thresholds as `fixed_data`.
- Always return `can_continue=True`.


### Data quality guardrails

Approved active DQ rules from `03_review` are read from `METADATA_DQ_RULES` through the configured metadata route. They use the same guardrail result contract as schema and data-change checks:

- `status`: `passed`, `warning`, or `failed`
- `can_continue`: whether the notebook can publish the target
- `checks`: aggregate rule-level outcomes
- `message`: human-readable summary

Severity controls blocking behavior. A failing `error` rule returns `failed` with `can_continue=False` and `stop_if_failed(...)` blocks before the target write. A failing `warning` rule returns `warning` with `can_continue=True`, so the notebook writes the full `df_output` and tags warning-failed rows with `_dq_check_status` and `_dq_failed_rules`. Passing rules also write the full `df_output` with DQ technical columns. v1 does not write row-level failure metadata, filter invalid rows out of the target, send alerts, create exception datasets, or perform partial target writes. Aggregated DQ results can feed dashboards and alerts later.

Presets determine baseline and enforcement behaviour. Overrides adjust thresholds only. The schema dictionaries and threshold overrides below are user-editable guardrails; `stop_if_failed(...)` only stops based on the result objects returned by `validate_schema(...)` and `monitor_data_changes(...)`, and `enforce_dq_rules(...)`.
Override only the values your pipeline needs:

```python
SOURCE_DATA_CHANGE_OVERRIDES = {
    "block_numeric_psi": 0.30,
}
```


In [ ]:
USE_SAMPLE_DATA = True

ENV_NAME = ENV
SOURCE_LAYER = "source"
TARGET_LAYER = "product"
SOURCE_KIND = "lakehouse"  # lakehouse | warehouse | csv | parquet | excel
TARGET_KIND = "lakehouse"  # lakehouse | warehouse
SOURCE_TABLE = "minimal_source" if USE_SAMPLE_DATA else "CHANGE_ME_source_table"
TARGET_TABLE = "sample_agreement_output" if USE_SAMPLE_DATA else "CHANGE_ME_target_table"
SOURCE_FILE_PATH = "Files/sample/minimal_source.csv" if USE_SAMPLE_DATA else "Files/CHANGE_ME/source_file.csv"
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "target_dataset"
TABLE_NAME = TARGET_TABLE
WRITE_MODE = "overwrite"
METADATA_WRITE_MODE = "append"
BUSINESS_KEYS = ["customer_id"]

SOURCE_EXPECTED_SCHEMA = {
    "customer_id": "bigint",
    "event_ts": "string",
    "status": "string",
    "amount": "double",
    "email": "string",
    "country_code": "string",
}
TARGET_EXPECTED_SCHEMA = {
    "customer_id": "bigint",
    "event_ts": "string",
    "status": "string",
    "amount": "double",
    "email": "string",
    "country_code": "string",
    "amount_band": "string",
}

SOURCE_SCHEMA_CHECK = "allow_new_columns"  # allow added source fields; block missing/type changes
TARGET_SCHEMA_CHECK = "strict"  # target business output must match this notebook's contract

SOURCE_DATA_CHANGE_CHECK = "changing_data"  # preset chooses source baseline and enforcement behaviour
TARGET_DATA_CHANGE_CHECK = "changing_data"  # preset chooses target baseline and enforcement behaviour
SOURCE_DATA_CHANGE_OVERRIDES = {
    # Leave empty to use FabricOps defaults.
    # "block_numeric_psi": 0.30,
    # "max_row_count_change_percent": 75,
}
TARGET_DATA_CHANGE_OVERRIDES = {
    # Leave empty to use FabricOps defaults.
    # "block_numeric_psi": 0.30,
    # "max_row_count_change_percent": 75,
}
DATA_CHANGE_COLUMNS = None  # optional distribution column allow-list
MARK_CURRENT_PROFILE_AS_APPROVED_BASELINE = False  # explicit promotion; not hidden in presets

PIPELINE_NAME = f"{SOURCE_TABLE}_to_{TARGET_TABLE}"
TOPIC = "production_pipeline"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S%f")
RUN_ID = f"{PIPELINE_NAME}_{ENV_NAME}_{EXECUTION_TIMESTAMP}"

CATALOGUE_TABLE = "METADATA_DATA_CATALOGUE"
LINEAGE_TABLE = "METADATA_DATA_LINEAGE_TABLE"
DQ_RULES_TABLE = "METADATA_DQ_RULES"
PIPELINE_RUN_TABLE = "METADATA_PIPELINE_RUN"

source_store = CONFIG.path_config.paths[ENV_NAME][SOURCE_LAYER]
target_store = CONFIG.path_config.paths[ENV_NAME][TARGET_LAYER]
metadata_store = CONFIG.path_config.paths[ENV_NAME]["metadata"]

runtime_context = {
    "run_id": RUN_ID,
    "environment": ENV_NAME,
    "dataset_name": DATASET_NAME,
    "source_kind": SOURCE_KIND,
    "target_kind": TARGET_KIND,
    "source_table": SOURCE_TABLE,
    "target_table": TARGET_TABLE,
    "pipeline_name": PIPELINE_NAME,
}


## 4. Data agreement selector and notebook registration

Select the data agreement and expose the notebook registration context for lineage and catalogue evidence. The selector registers this notebook as `02_pipeline` through the shared metadata route when the user clicks the registration button.

The selector widget records the active notebook through the metadata route. The lookup below reads active notebook registrations and uses `registration_id` when available.


In [ ]:
agreement_selector = widget_select_agreement(
    CONFIG,
    ENV_NAME,
    spark_session=spark,
    register_notebook=True,
    notebook_type="02_pipeline",
    environment_name=ENV_NAME,
    dataset_name=DATASET_NAME,
    table_name=TABLE_NAME,
    topic=TOPIC,
    pipeline_name=PIPELINE_NAME,
)

selected_agreement = get_selected_agreement()
AGREEMENT_ID = str(selected_agreement.get("agreement_id") or agreement_selector.value or "")
AGREEMENT_CONTRACT_VERSION = str(selected_agreement.get("contract_version") or "")

# Agreement selection registers this notebook when register_notebook=True.
# Keep registry identifiers optional here so pipeline cells stay focused on the approved agreement.
NOTEBOOK_REGISTRY_ID = selected_agreement.get("registration_id")
NOTEBOOK_ID = selected_agreement.get("notebook_id")

registration_context = {
    "agreement_id": AGREEMENT_ID,
    "agreement_contract_version": AGREEMENT_CONTRACT_VERSION,
    "notebook_registry_id": NOTEBOOK_REGISTRY_ID,
    "notebook_id": NOTEBOOK_ID,
}
display(registration_context)


## 5. Source read examples

Choose one active path with `SOURCE_KIND`. The lakehouse and warehouse examples use table helpers. The CSV and Parquet examples use Fabric lakehouse `Files/...` paths and are realistic for landing-zone or reference-file inputs.


In [ ]:
# Option A: Fabric lakehouse Delta table.
if SOURCE_KIND == "lakehouse":
    df_source = read_lakehouse_table(CONFIG, ENV_NAME, SOURCE_LAYER, SOURCE_TABLE, spark_session=spark)

# Option B: Fabric warehouse table.
elif SOURCE_KIND == "warehouse":
    df_source = read_warehouse_table(CONFIG, ENV_NAME, SOURCE_LAYER, "dbo", SOURCE_TABLE, spark_session=spark)

# Option C: CSV file from a lakehouse Files path.
elif SOURCE_KIND == "csv":
    df_source = read_lakehouse_csv(CONFIG, ENV_NAME, SOURCE_LAYER, SOURCE_FILE_PATH, spark_session=spark, header=True)

# Option D: Parquet file or folder from a lakehouse Files path.
elif SOURCE_KIND == "parquet":
    df_source = read_lakehouse_parquet(CONFIG, ENV_NAME, SOURCE_LAYER, SOURCE_FILE_PATH, spark_session=spark)

# Option E: Excel file from a lakehouse Files path.
elif SOURCE_KIND == "excel":
    df_source = read_lakehouse_excel(CONFIG, ENV_NAME, SOURCE_LAYER, SOURCE_FILE_PATH, spark_session=spark)

else:
    raise ValueError(f"Unsupported SOURCE_KIND: {SOURCE_KIND}")


## 6. Validate source schema

The expected schemas above are pipeline-specific configuration for this base `02_pipeline`. They intentionally stay local and simple: check the source after read, then check the transformed target shape before runtime audit columns are added. Governance workflows can enhance this later, but this template does not use widgets, metadata tables, approval workflows, contract versioning, or evidence persistence for schema checks.


In [ ]:
source_schema_result = validate_schema(
    dataframe=df_source,
    expected_schema=SOURCE_EXPECTED_SCHEMA,
    preset=SOURCE_SCHEMA_CHECK,
)
source_schema_result


## 7. Monitor source data changes

Profile the selected source DataFrame and display the profile before transformation. This creates reusable evidence without applying governance enforcement.


In [ ]:
source_row_count = df_source.count()
source_change_result = monitor_data_changes(
    spark=spark,
    dataframe=df_source,
    metadata_table=CATALOGUE_TABLE,
    dataset_name=DATASET_NAME,
    table_name=SOURCE_TABLE,
    stage="source",
    preset=SOURCE_DATA_CHANGE_CHECK,
    exclude_run_id=RUN_ID,
    distribution_columns=DATA_CHANGE_COLUMNS,
    policy_overrides=SOURCE_DATA_CHANGE_OVERRIDES,
)
source_profile = source_change_result["profile"]
source_drift = source_change_result["result"]
print(f"Source data-change monitoring: {source_drift['status']}")

stop_if_failed(source_schema_result)
stop_if_failed(source_change_result)
display(source_profile)


## 8. Prepare source catalogue evidence

Prepare source profile evidence now, but defer the metadata write until the guarded pipeline path has reached the successful evidence-write point. Do not create target-specific profile tables.


In [ ]:
def enrich_profile_for_catalogue(profile_df, *, evidence_role, profile_stage, table_name, layer, asset_kind, row_count, source_change_signal=None, dq_summary=None):
    baseline_status = "approved" if MARK_CURRENT_PROFILE_AS_APPROVED_BASELINE else "observed"
    dq_summary = dq_summary or {}
    return (
        profile_df
        .withColumn("AGREEMENT_ID", F.lit(AGREEMENT_ID))
        .withColumn("AGREEMENT_CONTRACT_VERSION", F.lit(AGREEMENT_CONTRACT_VERSION))
        .withColumn("NOTEBOOK_REGISTRY_ID", F.lit(NOTEBOOK_REGISTRY_ID))
        .withColumn("NOTEBOOK_ID", F.lit(NOTEBOOK_ID))
        .withColumn("PROFILE_RUN_ID", F.lit(RUN_ID))
        .withColumn("ENVIRONMENT_NAME", F.lit(ENV_NAME))
        .withColumn("DATASET_NAME", F.lit(DATASET_NAME))
        .withColumn("PIPELINE_NAME", F.lit(PIPELINE_NAME))
        .withColumn("EVIDENCE_ROLE", F.lit(evidence_role))
        .withColumn("PROFILE_STAGE", F.lit(profile_stage))
        .withColumn("PROFILE_STATUS", F.lit("successful"))
        .withColumn("BASELINE_STATUS", F.lit(baseline_status))
        .withColumn("SOURCE_SCHEMA_CHECK", F.lit(SOURCE_SCHEMA_CHECK))
        .withColumn("TARGET_SCHEMA_CHECK", F.lit(TARGET_SCHEMA_CHECK))
        .withColumn("SOURCE_DATA_CHANGE_CHECK", F.lit(SOURCE_DATA_CHANGE_CHECK))
        .withColumn("TARGET_DATA_CHANGE_CHECK", F.lit(TARGET_DATA_CHANGE_CHECK))
        .withColumn("SOURCE_CHANGE_SIGNAL_JSON", F.lit(json.dumps(source_change_signal, default=str) if source_change_signal is not None else None))
        .withColumn("DQ_STATUS", F.lit(dq_summary.get("DQ_STATUS")))
        .withColumn("DQ_RULE_COUNT", F.lit(dq_summary.get("DQ_RULE_COUNT", 0)))
        .withColumn("DQ_FAILED_RULE_COUNT", F.lit(dq_summary.get("DQ_FAILED_RULE_COUNT", 0)))
        .withColumn("DQ_WARNING_RULE_COUNT", F.lit(dq_summary.get("DQ_WARNING_RULE_COUNT", 0)))
        .withColumn("DQ_ERROR_RULE_COUNT", F.lit(dq_summary.get("DQ_ERROR_RULE_COUNT", 0)))
        .withColumn("DQ_FAILED_ROW_COUNT", F.lit(dq_summary.get("DQ_FAILED_ROW_COUNT", 0)))
        .withColumn("DQ_FAILED_ROW_PERCENT", F.lit(dq_summary.get("DQ_FAILED_ROW_PERCENT", 0.0)))
        .withColumn("DQ_CHECKED_AT", F.lit(dq_summary.get("DQ_CHECKED_AT")))
        .withColumn("LAYER", F.lit(layer))
        .withColumn("ASSET_KIND", F.lit(asset_kind))
        .withColumn("PROFILED_TABLE_NAME", F.lit(table_name))
        .withColumn("PROFILED_ROW_COUNT", F.lit(row_count))
        .withColumn("environment_name", F.lit(ENV_NAME))
        .withColumn("dataset_name", F.lit(DATASET_NAME))
        .withColumn("table_name", F.lit(table_name))
        .withColumn("column_name", F.col("COLUMN_NAME"))
        .withColumn("layer", F.lit(layer))
        .withColumn("asset_kind", F.lit(asset_kind))
        .withColumn("pipeline_name", F.lit(PIPELINE_NAME))
        .withColumn("profile_run_id", F.lit(RUN_ID))
        .withColumn("profile_stage", F.lit(profile_stage))
        .withColumn("profile_status", F.lit("success"))
        .withColumn("baseline_status", F.lit(baseline_status))
        .withColumn("source_data_change_check", F.lit(SOURCE_DATA_CHANGE_CHECK))
        .withColumn("profile_baseline_mode", F.lit("approved" if MARK_CURRENT_PROFILE_AS_APPROVED_BASELINE else "observed"))
        .withColumn("data_type", F.col("DATA_TYPE"))
        .withColumn("row_count", F.col("ROW_COUNT"))
        .withColumn("null_count", F.col("NULL_COUNT"))
        .withColumn("distinct_count", F.col("DISTINCT_COUNT"))
        .withColumn("distribution_type", F.col("DISTRIBUTION_TYPE") if "DISTRIBUTION_TYPE" in profile_df.columns else F.lit(None))
        .withColumn("distribution_json", F.col("DISTRIBUTION_JSON") if "DISTRIBUTION_JSON" in profile_df.columns else F.lit(None))
        .withColumn("profiled_at", F.col("RUN_TIMESTAMP").cast("string"))
        .withColumn("metadata_table_key", F.sha2(F.lower(F.concat_ws("|", F.lit(ENV_NAME), F.lit(DATASET_NAME), F.lit(table_name))), 256))
        .withColumn("metadata_column_key", F.sha2(F.lower(F.concat_ws("|", F.lit(ENV_NAME), F.lit(DATASET_NAME), F.lit(table_name), F.col("COLUMN_NAME"))), 256))
    )

source_catalogue_evidence = enrich_profile_for_catalogue(
    source_profile,
    evidence_role="source_profile",
    profile_stage="source",
    table_name=SOURCE_TABLE,
    layer=SOURCE_LAYER,
    asset_kind=SOURCE_KIND,
    row_count=source_row_count,
    source_change_signal=None,
)
source_catalogue_write_status = "pending_successful_pipeline"


## 9. Pipeline-specific transformations

Replace this section with real transformation logic for your data product. Keep this base template deterministic and technical; do not mix governance enforcement into this section.


In [ ]:
df_transformed = df_source

if "status" in df_transformed.columns:
    df_transformed = df_transformed.withColumn("status", F.trim(F.lower(F.col("status"))))

if "email" in df_transformed.columns:
    df_transformed = df_transformed.withColumn("email", F.trim(F.lower(F.col("email"))))

if "amount" in df_transformed.columns:
    df_transformed = df_transformed.withColumn("amount", F.col("amount").cast("double"))
    df_transformed = df_transformed.withColumn("amount_band", F.when(F.col("amount") >= 100, F.lit("high")).otherwise(F.lit("standard")))


## 10. Validate proposed target schema

Check the transformed target DataFrame before runtime audit columns are added so the expected target schema describes business columns only.


In [ ]:
target_schema_result = validate_schema(
    dataframe=df_transformed,
    expected_schema=TARGET_EXPECTED_SCHEMA,
    preset=TARGET_SCHEMA_CHECK,
)
target_schema_result


## 11. Runtime audit columns

Runtime audit columns are added inline with `withColumn(...)` before the target write. These columns answer:

- which run produced the row: `_pipeline_run_id`
- which pipeline produced it: `_pipeline_name`
- which environment produced it: `_pipeline_environment`
- which source table it came from: `_source_table`
- when it was loaded: `_record_loaded_timestamp`
- which notebook produced it: `_notebook_name`
- who or what ran it: `_loaded_by`

Audit columns are always useful. Hash columns are only for deduplication, masked key comparison, slowly changing dimensions, or change detection. Datetime feature columns are analytics features, not audit fields. Bucket columns are only for advanced large-table layout or skew handling. For simple parallel data loading, use `repartition_by`. For physical Delta pruning, use `partition_by` with a natural column.


In [ ]:
df_output = (
    df_transformed
    .withColumn("_pipeline_run_id", F.lit(RUN_ID))
    .withColumn("_pipeline_name", F.lit(PIPELINE_NAME))
    .withColumn("_pipeline_environment", F.lit(ENV_NAME))
    .withColumn("_source_table", F.lit(SOURCE_TABLE))
    .withColumn("_record_loaded_timestamp", F.current_timestamp())
)


## 12. Monitor target changes, enforce DQ, and publish

Evaluate target drift and approved active DQ rules before writing the selected target. The DQ guardrail mirrors the schema and data-change pattern: run the check, print the result, then call `stop_if_failed(...)`. Warning severity continues and writes the full annotated dataset; error severity stops before any target write. There is no row filtering, exception dataset, or partial write path in v1. Lakehouse and warehouse output examples are both shown, with the active path controlled by `TARGET_KIND`.


In [ ]:
target_row_count = df_output.count()
target_change_result = monitor_data_changes(
    spark=spark,
    dataframe=df_output,
    metadata_table=CATALOGUE_TABLE,
    dataset_name=DATASET_NAME,
    table_name=TARGET_TABLE,
    stage="target",
    preset=TARGET_DATA_CHANGE_CHECK,
    exclude_run_id=RUN_ID,
    distribution_columns=DATA_CHANGE_COLUMNS,
    policy_overrides=TARGET_DATA_CHANGE_OVERRIDES,
)
output_profile = target_change_result["profile"]
target_drift = target_change_result["result"]
print(f"Target data-change monitoring: {target_drift['status']}")

# DQ guardrail: approved active rules from 03_review run like schema and data drift checks.
dq_result = enforce_dq_rules(
    df_output,
    CONFIG,
    ENV_NAME,
    DATASET_NAME,
    TARGET_TABLE,
    spark_session=spark,
)
print(dq_result)

stop_if_failed(target_schema_result)
stop_if_failed(target_change_result)
# Warning severity writes full data with DQ technical annotations; error severity stops before write.
# No quarantine/filtering in v1.
stop_if_failed(dq_result)

# Use the dataframe returned by DQ enforcement so passed/warning runs keep technical annotation columns.
df_output = dq_result["dataframe"]

if TARGET_KIND == "lakehouse":
    write_lakehouse_table(df_output, CONFIG, ENV_NAME, TARGET_LAYER, TARGET_TABLE, mode=WRITE_MODE)
elif TARGET_KIND == "warehouse":
    write_warehouse_table(df_output, CONFIG, ENV_NAME, TARGET_LAYER, "dbo", TARGET_TABLE, mode=WRITE_MODE)
else:
    raise ValueError(f"Unsupported TARGET_KIND: {TARGET_KIND}")


## Optional large table write pattern

For larger lakehouse tables, keep audit columns lightweight and tune the write itself. Simple parallel loading should use `repartition_by`, not framework bucket columns.

- `partition_by` controls the physical Delta table layout. Use natural pruning columns such as `event_date`, `ingestion_date`, `batch_date`, `source_file_date`, or `year_month`.
- `repartition_by` controls Spark dataframe parallelism before writing. Use column-based repartitioning for common medium/large writes, or an integer partition count for very large writes after testing.
- Inspect natural partition sizes with `groupBy(...).count()` before committing to a physical partition layout.
- `repartition_by=2000` means Spark creates roughly 2000 dataframe partitions before writing. This is a tuning starting point, not a universal rule. Adjust it based on cluster size, target file size, skew, and write performance.
- Hash bucket columns are an advanced local recipe only for huge or skewed natural partitions. Keep them out of the standard path and do not make them a reusable helper.


In [ ]:
# Optional inspection: check natural partition sizes before choosing a physical partition column.
# df_output.groupBy("event_date").count().orderBy(F.desc("count")).display()

# Optional example: large table with natural Delta partition columns and column-based Spark repartitioning.
# LARGE_TABLE_PARTITION_COLUMNS = ["event_date"]
# LARGE_TABLE_REPARTITION_BY = ["event_date"]
#
# write_lakehouse_table(
#     df_output,
#     CONFIG,
#     ENV_NAME,
#     TARGET_LAYER,
#     TARGET_TABLE,
#     mode=WRITE_MODE,
#     partition_by=LARGE_TABLE_PARTITION_COLUMNS,
#     repartition_by=LARGE_TABLE_REPARTITION_BY,
# )

# Optional example: very large table with an explicit dataframe partition count.
# LARGE_TABLE_PARTITION_COLUMNS = ["event_date"]
# LARGE_TABLE_REPARTITION_BY = 2000
#
# write_lakehouse_table(
#     df_output,
#     CONFIG,
#     ENV_NAME,
#     TARGET_LAYER,
#     TARGET_TABLE,
#     mode=WRITE_MODE,
#     partition_by=LARGE_TABLE_PARTITION_COLUMNS,
#     repartition_by=LARGE_TABLE_REPARTITION_BY,
# )

# Optional advanced recipe only for huge or skewed natural partitions:
# create a local hash bucket expression in this notebook and include it in partition_by.
# Do this only after natural partition columns are still too large or too skewed.
# HASH_BUCKET_COUNT = 32
# df_large = df_output.withColumn("_write_hash_bucket", F.pmod(F.abs(F.hash(F.col(BUSINESS_KEYS[0]))), F.lit(HASH_BUCKET_COUNT)))
# write_lakehouse_table(
#     df_large,
#     CONFIG,
#     ENV_NAME,
#     TARGET_LAYER,
#     TARGET_TABLE,
#     mode=WRITE_MODE,
#     partition_by=["event_date", "_write_hash_bucket"],
#     repartition_by=2000,
# )


## 13. Optional: read back output

Read the written target table back for inspection if needed. Target drift was already evaluated against `df_output` before publication so blocking drift can stop the write.


In [ ]:
if TARGET_KIND == "lakehouse":
    df_published = read_lakehouse_table(CONFIG, ENV_NAME, TARGET_LAYER, TARGET_TABLE, spark_session=spark)
else:
    df_published = read_warehouse_table(CONFIG, ENV_NAME, TARGET_LAYER, "dbo", TARGET_TABLE, spark_session=spark)


## 14. Display target profile

Display the target profile that was evaluated before publication.


In [ ]:
display(output_profile)


## 15. Write output catalogue evidence

Write output profile evidence into the same reusable metadata catalogue pattern as the source evidence.


In [ ]:
output_catalogue_evidence = enrich_profile_for_catalogue(
    output_profile,
    evidence_role="output_profile",
    profile_stage="target",
    table_name=TARGET_TABLE,
    layer=TARGET_LAYER,
    asset_kind=TARGET_KIND,
    row_count=target_row_count,
    source_change_signal=None,
    dq_summary=dq_result.get("summary"),
)

write_lakehouse_table(
    source_catalogue_evidence,
    CONFIG,
    ENV_NAME,
    "metadata",
    CATALOGUE_TABLE,
    mode=METADATA_WRITE_MODE,
)
source_catalogue_write_status = "written"

write_lakehouse_table(
    output_catalogue_evidence,
    CONFIG,
    ENV_NAME,
    "metadata",
    CATALOGUE_TABLE,
    mode=METADATA_WRITE_MODE,
)
output_catalogue_write_status = "written"


## 16. Write lineage

Build source-to-target lineage records from consumed sources and produced targets. The records are tied to the selected agreement, run id, pipeline name, environment, and notebook registry id when available, then written to the metadata lineage table.


In [ ]:
lineage_records = build_lineage_records(
    dataset_name=DATASET_NAME,
    run_id=RUN_ID,
    source_tables=[SOURCE_TABLE],
    target_table=TARGET_TABLE,
    transformation_steps=[
        {
            "step": 1,
            "source": SOURCE_TABLE,
            "target": TARGET_TABLE,
            "operation": "deterministic transform + runtime audit columns",
        },
    ],
)

captured_at = datetime.now(timezone.utc).isoformat()
lineage_rows = []
for row in lineage_records:
    payload = dict(row)
    payload.update({"agreement_id": AGREEMENT_ID, "notebook_registry_id": NOTEBOOK_REGISTRY_ID})
    lineage_rows.append({
        "lineage_id": f"{RUN_ID}_{row.get('step')}",
        "run_id": RUN_ID,
        "agreement_id": AGREEMENT_ID,
        "agreement_contract_version": AGREEMENT_CONTRACT_VERSION,
        "environment_name": ENV_NAME,
        "dataset_name": DATASET_NAME,
        "pipeline_name": PIPELINE_NAME,
        "source_table": row.get("source") or SOURCE_TABLE,
        "target_table": row.get("target") or TARGET_TABLE,
        "notebook_registry_id": NOTEBOOK_REGISTRY_ID,
        "notebook_id": NOTEBOOK_ID,
        "lineage_level": "table",
        "transformation_type": "deterministic_pipeline",
        "transformation_summary": row.get("operation"),
        "captured_at": captured_at,
        "lineage_payload_json": json.dumps(payload, default=str),
    })

lineage_df = spark.createDataFrame(lineage_rows)
write_lakehouse_table(lineage_df, CONFIG, ENV_NAME, "metadata", LINEAGE_TABLE, mode=METADATA_WRITE_MODE)
lineage_write_status = "written"


pipeline_run_row = {
    "run_id": RUN_ID,
    "agreement_id": AGREEMENT_ID,
    "agreement_contract_version": AGREEMENT_CONTRACT_VERSION,
    "environment_name": ENV_NAME,
    "dataset_name": DATASET_NAME,
    "pipeline_name": PIPELINE_NAME,
    "source_table": SOURCE_TABLE,
    "target_table": TARGET_TABLE,
    "source_row_count": source_row_count,
    "target_row_count": target_row_count,
    "source_schema_status": source_schema_result.get("status"),
    "target_schema_status": target_schema_result.get("status"),
    "source_drift_status": source_drift.get("status"),
    "target_drift_status": target_drift.get("status"),
    "dq_status": dq_result.get("status"),
    "dq_can_continue": bool(dq_result.get("can_continue")),
    "dq_rule_count": int(dq_result.get("summary", {}).get("DQ_RULE_COUNT", 0)),
    "dq_failed_rule_count": int(dq_result.get("summary", {}).get("DQ_FAILED_RULE_COUNT", 0)),
    "dq_warning_rule_count": int(dq_result.get("summary", {}).get("DQ_WARNING_RULE_COUNT", 0)),
    "dq_error_rule_count": int(dq_result.get("summary", {}).get("DQ_ERROR_RULE_COUNT", 0)),
    "dq_failed_row_count": int(dq_result.get("summary", {}).get("DQ_FAILED_ROW_COUNT", 0)),
    "schema_drift_status": "failed" if source_schema_result.get("status") == "failed" or target_schema_result.get("status") == "failed" else "passed",
    "pipeline_status": "published",
    "notebook_registry_id": NOTEBOOK_REGISTRY_ID,
    "notebook_id": NOTEBOOK_ID,
    "started_at": EXECUTION_TIMESTAMP,
    "completed_at": captured_at,
    "evidence_json": json.dumps(
        {
            "source_schema_result": source_schema_result,
            "target_schema_result": target_schema_result,
            "source_drift": source_drift,
            "target_drift": target_drift,
            "dq_summary": dq_result.get("summary"),
        },
        default=str,
    ),
}
write_lakehouse_table(spark.createDataFrame([pipeline_run_row]), CONFIG, ENV_NAME, "metadata", PIPELINE_RUN_TABLE, mode=METADATA_WRITE_MODE)
pipeline_run_write_status = "written"


## 17. Summary

Display a concise run summary for review and operational support.


In [ ]:
run_summary = {
    "run_id": RUN_ID,
    "pipeline_name": PIPELINE_NAME,
    "environment": ENV_NAME,
    "source_table": SOURCE_TABLE,
    "source_row_count": source_row_count,
    "source_schema_check": SOURCE_SCHEMA_CHECK,
    "target_schema_check": TARGET_SCHEMA_CHECK,
    "source_data_change_check": SOURCE_DATA_CHANGE_CHECK,
    "target_data_change_check": TARGET_DATA_CHANGE_CHECK,
    "source_drift_status": source_drift.get("status"),
    "target_table": TARGET_TABLE,
    "target_row_count": target_row_count,
    "target_drift_status": target_drift.get("status"),
    "dq_status": dq_result.get("status"),
    "dq_can_continue": dq_result.get("can_continue"),
    "dq_rule_count": len(dq_result.get("checks", [])),
    "selected_agreement": AGREEMENT_ID,
    "agreement_contract_version": AGREEMENT_CONTRACT_VERSION,
    "notebook_registry_id": NOTEBOOK_REGISTRY_ID,
    "source_catalogue_write_status": source_catalogue_write_status,
    "output_catalogue_write_status": output_catalogue_write_status,
    "lineage_write_status": lineage_write_status,
    "pipeline_run_write_status": pipeline_run_write_status,
    "catalogue_table": CATALOGUE_TABLE,
    "lineage_table": LINEAGE_TABLE,
    "dq_rules_table": DQ_RULES_TABLE,
    "pipeline_run_table": PIPELINE_RUN_TABLE,
}
display(run_summary)
